In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import sqlite3
import plotly.express as px

caminho_do_arquivo = 'netflix_titles.csv'

df = pd.read_csv(caminho_do_arquivo)

# Criando um Banco de Dados SQL virtual para fazermos as consultas
conn = sqlite3.connect(':memory:')
df.to_sql('netflix', conn, index=False)

print("Base carregada do Google Drive e Banco de Dados criado com sucesso!")

Base carregada do Google Drive e Banco de Dados criado com sucesso!


In [18]:
# 1. Escrevendo a nossa consulta (Query) em SQL
query = """
    SELECT type, COUNT(*) as quantidade
    FROM netflix
    GROUP BY type
"""
resultado_sql = pd.read_sql(query, conn)

# Mostrando a tabela de texto na tela
print("Tabela de Resultados (SQL):")
display(resultado_sql)

# 3. Criando um gráfico de pizza
grafico = px.pie(
    resultado_sql,
    values='quantidade',
    names='type',
    title='Proporção do Catálogo: Filmes vs Séries na Netflix',
    color_discrete_sequence=['#004766', '#564d4d']
)


grafico.show()

Tabela de Resultados (SQL):


,type,quantidade
0,Movie,6131
1,TV Show,2676


In [19]:
# 1. Consulta SQL: Contando a quantidade de títulos por ano de lançamento
query_anos = """
    SELECT release_year, COUNT(*) as quantidade
    FROM netflix
    GROUP BY release_year
    ORDER BY release_year
"""
resultado_anos = pd.read_sql(query_anos, conn)

# 3. Criando o Gráfico de Linha
grafico_linha = px.line(
    resultado_anos,
    x='release_year',
    y='quantidade',
    title='Evolução de Lançamentos na Netflix por Ano',
    markers=True,
    color_discrete_sequence=['#E50914']
)

# Mostrando o gráfico na tela
grafico_linha.show()

In [11]:
# 1. Consulta SQL: Agrupando as categorias, contando e pegando o Top 10
query_generos = """
    SELECT listed_in as categoria, COUNT(*) as quantidade
    FROM netflix
    GROUP BY categoria
    ORDER BY quantidade DESC
    LIMIT 10
"""
resultado_generos = pd.read_sql(query_generos, conn)

grafico_barras = px.bar(
    resultado_generos,
    x='quantidade',
    y='categoria',
    title='Top 10 Categorias com Mais Títulos na Netflix',
    orientation='h',
    color='quantidade',
    color_continuous_scale='Blues'
)

# Invertendo o eixo Y para a maior barra ficar no topo
grafico_barras.update_layout(yaxis={'categoryorder':'total ascending'})

grafico_barras.show()

In [12]:
# 1. Separando os gêneros por vírgula e criando uma lista na célula
df['generos_separados'] = df['listed_in'].str.split(', ')

# 2. Usando o explode para criar uma linha individual para cada gênero
df_explodido = df.explode('generos_separados')

# 3. Contando os gêneros individuais e pegando o Top 10
top_10_real = df_explodido['generos_separados'].value_counts().head(10).reset_index()
top_10_real.columns = ['categoria', 'quantidade']

grafico_barras_corrigido = px.bar(
    top_10_real,
    x='quantidade',
    y='categoria',
    title='Top 10 Gêneros Individuais com Mais Títulos na Netflix (Dado Limpo)',
    orientation='h',
    color='quantidade',
    color_continuous_scale='Blues'
)

grafico_barras_corrigido.update_layout(yaxis={'categoryorder':'total ascending'})

grafico_barras_corrigido.show()